In [1]:
include("GenX.jl")
using .GenX
using HiGHS
using JuMP
using Gurobi
using DataFrames

  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: 0.4.1


┌ Info: Running precompile script for GenX. This may take a few minutes.
└ @ Main.GenX c:\Users\mjarada1\Desktop\GenX.jl\src\startup\genx_startup.jl:56


  ____           __  __   _ _
 / ___| ___ _ __ \ \/ /  (_) |
| |  _ / _ \ '_ \ \  /   | | |
| |_| |  __/ | | |/  \ _ | | |
 \____|\___|_| |_/_/\_(_)/ |_|
                       |__/
 Version: nothing


In [2]:
# Genx case_runners file,
case = "..\\example_systems\\15_markets_prm_ro"
settings_path = joinpath(case, "settings")
policies_path = joinpath(case, "policies")
output_folder = joinpath(case, "Results") # Write-output settings YAML file path,
genx_settings = joinpath(settings_path, "genx_settings.yml") # Settings YAML file path,
mysetup = GenX.configure_settings(genx_settings, output_folder) # mysetup dictionary stores settings and GenX-specific parameters,
optimizer = Gurobi.Optimizer
OPTIMIZER =  GenX.configure_solver(settings_path, optimizer)



Configuring Settings


MathOptInterface.OptimizerWithAttributes(Gurobi.Optimizer, Pair{MathOptInterface.AbstractOptimizerAttribute, Any}[MathOptInterface.RawOptimizerAttribute("FeasibilityTol") => 1.0e-6, MathOptInterface.RawOptimizerAttribute("PreDual") => 0, MathOptInterface.RawOptimizerAttribute("Method") => 2, MathOptInterface.RawOptimizerAttribute("Crossover") => 0, MathOptInterface.RawOptimizerAttribute("TimeLimit") => 1100000, MathOptInterface.RawOptimizerAttribute("OutputFlag") => 1, MathOptInterface.RawOptimizerAttribute("NumericFocus") => 0, MathOptInterface.RawOptimizerAttribute("BarHomogeneous") => 1, MathOptInterface.RawOptimizerAttribute("MIPGap") => 0.0001, MathOptInterface.RawOptimizerAttribute("OptimalityTol") => 1.0e-6, MathOptInterface.RawOptimizerAttribute("AggFill") => 10, MathOptInterface.RawOptimizerAttribute("BarConvTol") => 0.0001, MathOptInterface.RawOptimizerAttribute("Presolve") => 1])

In [3]:
TDRpath = joinpath(case, mysetup["TimeDomainReductionFolder"])

"..\\example_systems\\15_markets_prm_ro\\TDR_results"

In [4]:
system_path = joinpath(case, mysetup["SystemFolder"])

"..\\example_systems\\15_markets_prm_ro\\system"

In [6]:
GenX.prevent_doubled_timedomainreduction(system_path)



In [ ]:
myinputs =  GenX.load_inputs(mysetup, case)
EP =  GenX.generate_model(mysetup, myinputs, OPTIMIZER)
EP, solve_time =  GenX.solve_model(EP, mysetup)
myinputs["solve_time"] = solve_time
inputs = myinputs
setup = mysetup

In [ ]:
Morris_range = load_dataframe(joinpath(case, "Method_of_morris_range.csv"))
groups = Morris_range[!, :Group]
p_steps = Morris_range[!, :p_steps]
total_num_trajectory = Morris_range[!, :total_num_trajectory][1]
num_trajectory = Morris_range[!, :num_trajectory][1]
len_design_mat = Morris_range[!, :len_design_mat][1]
uncertain_columns = unique(Morris_range[!, :Parameter])
#save_parameters = zeros(length(Morris_range[!,:Parameter]))
gen = inputs["RESOURCES"]
sigma = zeros((1, 2))


In [ ]:
column = uncertain_columns[1]
col_sym = Symbol(lowercase(column))
# column_f is the function to get the value "column" for each generator
column_f = isdefined(GenX, col_sym) ? getfield(GenX, col_sym) :
           r -> getproperty(r, col_sym)

In [ ]:
column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100)

In [ ]:
sigma = [sigma;
             [column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100) column_f.(gen) .*
                    (1 .+
                     Morris_range[Morris_range[!, :Parameter] .== column,
                 :Upper_bound] ./ 100)]]

In [ ]:

for column in uncertain_columns
    col_sym = Symbol(lowercase(column))
    # column_f is the function to get the value "column" for each generator
    column_f = isdefined(GenX, col_sym) ? getfield(GenX, col_sym) :
               r -> getproperty(r, col_sym)
    sigma = [sigma;
             [column_f.(gen) .* (1 .+
               Morris_range[Morris_range[!, :Parameter] .== column, :Lower_bound] ./
               100) column_f.(gen) .*
                    (1 .+
                     Morris_range[Morris_range[!, :Parameter] .== column,
                 :Upper_bound] ./ 100)]]
end

In [ ]:

sigma = sigma[2:end, :]

p_range = mapslices(x -> [x], sigma, dims = 2)[:]

In [7]:
df_duals = DataFrame()
T = inputs["T"]

if inputs["ro_settings"]["MarketBuyPrices"] == 1
    Z = inputs["Z"] 
    MZ = inputs["MZ"]
    GenX.write_ro_market_buy(joinpath(case, "resxx"), inputs, setup, EP)
    for i in 1:Z
        if i in MZ
            df_duals[!, "S_MarketBuy_$(i)"] =  vec(dual.(EP[:cDualSmb][i,:]).data)
        else
            df_duals[!, "S_MarketBuy_$(i)"] = zeros(T)
        end
    end
end

In [ ]:
using DataFrames

# Create a sample DataFrame
df = DataFrame(A = 1:10, B = 10:19)

# Create a new column with fewer values
new_column = [100, 200, 300, 400, 500]

# Add the new column to the DataFrame
df.C = [new_column; fill(missing, nrow(df) - length(new_column))]

# If you want to replace missing values with a specific value (e.g., 0):
# df.C = coalesce.(df.C, 0)

println(df)